# Gate Package Tests
Tests for `gates.py` — covers single-qubit gates, two-qubit gates, channels, circuit utilities, and density-matrix conversions.

**Conventions used throughout:**
- Unitary gates map to *orthogonal* matrices in the Pauli basis: `M @ M.T == I`
- The first row/column of every gate matrix must be `[1, 0, 0, ...]` (trace preservation)
- Parameterised gates must recover the identity at `param = 0`
- Channels must be trace-preserving and (for non-unital ones) have a non-zero affine shift

In [1]:
import numpy as np
import jax.numpy as jnp
from itertools import product as iproduct

# ── adjust this import to match your file/module name ──────────────────────
from gates_jax import (
    Gate,
    circuit_matrix,
    calculate_steadystate_solve,
    density_matrix_to_pauli_vector,
    pauli_vector_to_density_matrix,
    pauli_basis_strings,
    string_to_vector,
    expand_gate_Bloch,
    pauli_matrices,
)
# ───────────────────────────────────────────────────────────────────────────

ATol = 1e-5  # absolute tolerance for all np.allclose checks

def check(condition, label):
    status = "✅ PASS" if condition else "❌ FAIL"
    print(f"{status}  {label}")
    return condition

print("Imports OK")

Imports OK


---
## 1  Shared helpers

In [2]:
def is_orthogonal(M, atol=ATol):
    """Check M @ M.T ≈ I  (real orthogonal)."""
    M = np.array(M, dtype=float)
    return np.allclose(M @ M.T, np.eye(M.shape[0]), atol=atol)

def preserves_trace(M, atol=ATol):
    """First row must be [1, 0, 0, …] — trace preservation."""
    M = np.array(M)
    return np.allclose(M[0, :], np.eye(M.shape[0])[0], atol=atol)

def first_col_ok(M, atol=ATol):
    """First column must be [1, 0, 0, …] for unital maps."""
    M = np.array(M)
    expected = np.zeros(M.shape[0])
    expected[0] = 1.0
    return np.allclose(M[:, 0], expected, atol=atol)

def get_matrix(name, params=None, N=1, loc=0):
    """Convenience wrapper — returns the raw (un-expanded) gate matrix."""
    return Gate(name, params=params)._get_matrix()

# Fix the helper name (underscore in method is fine here)
def mat(name, params=None):
    return np.array(Gate(name, params=params)._get_matrix())

print("Helpers defined")

Helpers defined


---
## 2  Single-qubit fixed gates

In [3]:
FIXED_1Q = ["H", "X", "Y", "Z", "T", "S"]
ALIASES  = [("H", "Hadamard"), ("X", "PauliX"), ("Y", "PauliY"), ("Z", "PauliZ")]

print("── Shape (4×4) ──────────────────────────")
for name in FIXED_1Q:
    M = mat(name)
    check(M.shape == (4, 4), f"{name}: shape is (4,4)")

print("── Orthogonality ────────────────────────")
for name in FIXED_1Q:
    check(is_orthogonal(mat(name)), f"{name}: M @ Mᵀ = I")

print("── Trace preservation (row 0) ───────────")
for name in FIXED_1Q:
    check(preserves_trace(mat(name)), f"{name}: trace-preserving")

print("── Unital (col 0) ───────────────────────")
for name in FIXED_1Q:
    check(first_col_ok(mat(name)), f"{name}: unital")

print("── Aliases give identical matrices ──────")
for canonical, alias in ALIASES:
    check(np.allclose(mat(canonical), mat(alias)), f"{alias} == {canonical}")

── Shape (4×4) ──────────────────────────
✅ PASS  H: shape is (4,4)
✅ PASS  X: shape is (4,4)
✅ PASS  Y: shape is (4,4)
✅ PASS  Z: shape is (4,4)
✅ PASS  T: shape is (4,4)
✅ PASS  S: shape is (4,4)
── Orthogonality ────────────────────────
✅ PASS  H: M @ Mᵀ = I
✅ PASS  X: M @ Mᵀ = I
✅ PASS  Y: M @ Mᵀ = I
✅ PASS  Z: M @ Mᵀ = I
✅ PASS  T: M @ Mᵀ = I
✅ PASS  S: M @ Mᵀ = I
── Trace preservation (row 0) ───────────
✅ PASS  H: trace-preserving
✅ PASS  X: trace-preserving
✅ PASS  Y: trace-preserving
✅ PASS  Z: trace-preserving
✅ PASS  T: trace-preserving
✅ PASS  S: trace-preserving
── Unital (col 0) ───────────────────────
✅ PASS  H: unital
✅ PASS  X: unital
✅ PASS  Y: unital
✅ PASS  Z: unital
✅ PASS  T: unital
✅ PASS  S: unital
── Aliases give identical matrices ──────
✅ PASS  Hadamard == H
✅ PASS  PauliX == X
✅ PASS  PauliY == Y
✅ PASS  PauliZ == Z


In [4]:
print("── Known special values ─────────────────")

# H² = I
H = mat("H")
check(np.allclose(H @ H, np.eye(4), atol=ATol), "H²  = I")

# X² = Y² = Z² = I
for name in ["X", "Y", "Z"]:
    M = mat(name)
    check(np.allclose(M @ M, np.eye(4), atol=ATol), f"{name}² = I")

# S² = Z
S = mat("S"); Z = mat("Z")
check(np.allclose(S @ S, Z, atol=ATol), "S²  = Z")

# T² = S
T = mat("T")
check(np.allclose(T @ T, S, atol=ATol), "T²  = S")

# H X H = Z  (conjugation)
X = mat("X")
check(np.allclose(H @ X @ H, Z, atol=ATol), "H X H = Z")

# H Z H = X
check(np.allclose(H @ Z @ H, X, atol=ATol), "H Z H = X")

── Known special values ─────────────────
✅ PASS  H²  = I
✅ PASS  X² = I
✅ PASS  Y² = I
✅ PASS  Z² = I
✅ PASS  S²  = Z
✅ PASS  T²  = S
✅ PASS  H X H = Z
✅ PASS  H Z H = X


True

---
## 3  Single-qubit parameterised gates

In [5]:
PARAM_1Q = ["RX", "RY", "RZ"]
TEST_ANGLES = [0.0, np.pi/6, np.pi/4, np.pi/3, np.pi/2, np.pi, 2*np.pi]

print("── Identity at param=0 ──────────────────")
for name in PARAM_1Q:
    M = mat(name, params=[0.0])
    check(np.allclose(M, np.eye(4), atol=ATol), f"{name}(0) = I")

print("── Orthogonality for all test angles ────")
for name in PARAM_1Q:
    failures = []
    for t in TEST_ANGLES:
        if not is_orthogonal(mat(name, params=[t])):
            failures.append(t)
    check(len(failures) == 0, f"{name}: orthogonal at angles {TEST_ANGLES} (failures: {failures})")

print("── 2π periodicity ───────────────────────")
for name in PARAM_1Q:
    for t in [0.3, 0.7, 1.2]:
        M0 = mat(name, params=[t])
        M2 = mat(name, params=[t + 2*np.pi])
        check(np.allclose(M0, M2, atol=ATol), f"{name}(t) = {name}(t+2π)  at t={t:.1f}")

── Identity at param=0 ──────────────────
✅ PASS  RX(0) = I
✅ PASS  RY(0) = I
✅ PASS  RZ(0) = I
── Orthogonality for all test angles ────
✅ PASS  RX: orthogonal at angles [0.0, 0.5235987755982988, 0.7853981633974483, 1.0471975511965976, 1.5707963267948966, 3.141592653589793, 6.283185307179586] (failures: [])
✅ PASS  RY: orthogonal at angles [0.0, 0.5235987755982988, 0.7853981633974483, 1.0471975511965976, 1.5707963267948966, 3.141592653589793, 6.283185307179586] (failures: [])
✅ PASS  RZ: orthogonal at angles [0.0, 0.5235987755982988, 0.7853981633974483, 1.0471975511965976, 1.5707963267948966, 3.141592653589793, 6.283185307179586] (failures: [])
── 2π periodicity ───────────────────────
✅ PASS  RX(t) = RX(t+2π)  at t=0.3
✅ PASS  RX(t) = RX(t+2π)  at t=0.7
✅ PASS  RX(t) = RX(t+2π)  at t=1.2
✅ PASS  RY(t) = RY(t+2π)  at t=0.3
✅ PASS  RY(t) = RY(t+2π)  at t=0.7
✅ PASS  RY(t) = RY(t+2π)  at t=1.2
✅ PASS  RZ(t) = RZ(t+2π)  at t=0.3
✅ PASS  RZ(t) = RZ(t+2π)  at t=0.7
✅ PASS  RZ(t) = RZ(t+2π)

/var/folders/qd/5q9cvk1x2lgcx1ddzjm2j60c0000gn/T/ipykernel_42610/2659351537.py:3: ComplexWarning: Casting complex values to real discards the imaginary part
  M = np.array(M, dtype=float)


In [6]:
print("── RX(π) = X in Bloch representation ────")
check(np.allclose(mat("RX", [np.pi]), mat("X"), atol=ATol), "RX(π) = X")

print("── RY(π) = Y ─────────────────────────────")
check(np.allclose(mat("RY", [np.pi]), mat("Y"), atol=ATol), "RY(π) = Y")

print("── RZ(π) = Z ─────────────────────────────")
check(np.allclose(np.real(mat("RZ", [np.pi])), mat("Z"), atol=ATol), "RZ(π) = Z")

print("── Rotation composition: RX(a)·RX(b) = RX(a+b) ──")
for a, b in [(0.3, 0.7), (np.pi/4, np.pi/3), (1.0, -0.5)]:
    LHS = mat("RX", [a]) @ mat("RX", [b])
    RHS = mat("RX", [a + b])
    check(np.allclose(LHS, RHS, atol=ATol), f"RX({a:.2f})·RX({b:.2f}) = RX({a+b:.2f})")

── RX(π) = X in Bloch representation ────
✅ PASS  RX(π) = X
── RY(π) = Y ─────────────────────────────
✅ PASS  RY(π) = Y
── RZ(π) = Z ─────────────────────────────
✅ PASS  RZ(π) = Z
── Rotation composition: RX(a)·RX(b) = RX(a+b) ──
✅ PASS  RX(0.30)·RX(0.70) = RX(1.00)
✅ PASS  RX(0.79)·RX(1.05) = RX(1.83)
✅ PASS  RX(1.00)·RX(-0.50) = RX(0.50)


In [7]:
print("── RY_Z: params required ─────────────────")
try:
    Gate("RY_Z")._get_matrix()
    check(False, "RY_Z without params should raise ValueError")
except ValueError:
    check(True, "RY_Z without params raises ValueError")

print("── RY_Z at π/2 approximates Hadamard ─────")
# RY_Z(π/2) = S · RY(π/2);  up to global phase this should match H
M = np.real(mat("RY_Z", [np.pi/2]))
check(is_orthogonal(M), "RY_Z(π/2) is orthogonal")

── RY_Z: params required ─────────────────
✅ PASS  RY_Z without params raises ValueError
── RY_Z at π/2 approximates Hadamard ─────
✅ PASS  RY_Z(π/2) is orthogonal


True

---
## 4  Two-qubit fixed gates

In [8]:
FIXED_2Q = ["CNOT", "SWAP"]

print("── Shape (16×16) ────────────────────────")
for name in FIXED_2Q:
    M = mat(name)
    check(M.shape == (16, 16), f"{name}: shape is (16,16)")

print("── Orthogonality ────────────────────────")
for name in FIXED_2Q:
    check(is_orthogonal(mat(name)), f"{name}: orthogonal")

print("── Trace preservation ───────────────────")
for name in FIXED_2Q:
    check(preserves_trace(mat(name)), f"{name}: trace-preserving")

print("── Self-inverse ─────────────────────────")
for name in FIXED_2Q:
    M = mat(name)
    check(np.allclose(M @ M, np.eye(16), atol=ATol), f"{name}² = I")

── Shape (16×16) ────────────────────────
✅ PASS  CNOT: shape is (16,16)
✅ PASS  SWAP: shape is (16,16)
── Orthogonality ────────────────────────
✅ PASS  CNOT: orthogonal
✅ PASS  SWAP: orthogonal
── Trace preservation ───────────────────
✅ PASS  CNOT: trace-preserving
✅ PASS  SWAP: trace-preserving
── Self-inverse ─────────────────────────
✅ PASS  CNOT² = I
✅ PASS  SWAP² = I


In [9]:
print("── CNOT: acts as identity on control-only observables ──")
# ZI should be invariant under CNOT (control qubit unaffected)
ZI = string_to_vector("ZI")
CNOT = mat("CNOT")
check(np.allclose(CNOT @ ZI, ZI, atol=ATol), "CNOT · |ZI⟩ = |ZI⟩")

# IX → XX under CNOT
IX = string_to_vector("IX")
XX = string_to_vector("XX")
check(np.allclose(CNOT @ IX, XX, atol=ATol), "CNOT · |IX⟩ = |XX⟩")

print("── SWAP: swaps XI ↔ IX ──────────────────")
XI = string_to_vector("XI")
SWAP = mat("SWAP")
check(np.allclose(SWAP @ XI, IX, atol=ATol), "SWAP · |XI⟩ = |IX⟩")
check(np.allclose(SWAP @ IX, XI, atol=ATol), "SWAP · |IX⟩ = |XI⟩")

── CNOT: acts as identity on control-only observables ──
✅ PASS  CNOT · |ZI⟩ = |ZI⟩
❌ FAIL  CNOT · |IX⟩ = |XX⟩
── SWAP: swaps XI ↔ IX ──────────────────
✅ PASS  SWAP · |XI⟩ = |IX⟩
✅ PASS  SWAP · |IX⟩ = |XI⟩


True

---
## 5  Two-qubit parameterised gates

In [10]:
PARAM_2Q = [
    ("RXX",      "IsingXX"),
    ("RYY",      "IsingYY"),
    ("RZZ",      "IsingZZ"),
    ("XXPlusYY",  None),
    ("XXMinusYY", None),
    ("CRX",       None),
    ("CRY",       None),
    ("CRZ",       None),
]

print("── Identity at param=0 ──────────────────")
for canonical, alias in PARAM_2Q:
    M = np.real(mat(canonical, [0.0]))
    check(np.allclose(M, np.eye(16), atol=ATol), f"{canonical}(0) = I")

print("── Orthogonality at several angles ──────")
for canonical, _ in PARAM_2Q:
    failures = []
    for t in [0.1, np.pi/4, np.pi/2, np.pi]:
        M = np.real(mat(canonical, [t]))
        if not is_orthogonal(M):
            failures.append(round(t, 3))
    check(len(failures) == 0, f"{canonical}: orthogonal (failures: {failures})")

print("── Aliases match ────────────────────────")
for canonical, alias in PARAM_2Q:
    if alias:
        for t in [0.5, 1.2]:
            check(
                np.allclose(np.real(mat(canonical, [t])), np.real(mat(alias, [t])), atol=ATol),
                f"{alias}({t}) == {canonical}({t})"
            )

── Identity at param=0 ──────────────────
✅ PASS  RXX(0) = I
✅ PASS  RYY(0) = I
✅ PASS  RZZ(0) = I
✅ PASS  XXPlusYY(0) = I
✅ PASS  XXMinusYY(0) = I
✅ PASS  CRX(0) = I
✅ PASS  CRY(0) = I
✅ PASS  CRZ(0) = I
── Orthogonality at several angles ──────
✅ PASS  RXX: orthogonal (failures: [])
✅ PASS  RYY: orthogonal (failures: [])
✅ PASS  RZZ: orthogonal (failures: [])
✅ PASS  XXPlusYY: orthogonal (failures: [])
✅ PASS  XXMinusYY: orthogonal (failures: [])
✅ PASS  CRX: orthogonal (failures: [])
✅ PASS  CRY: orthogonal (failures: [])
✅ PASS  CRZ: orthogonal (failures: [])
── Aliases match ────────────────────────
✅ PASS  IsingXX(0.5) == RXX(0.5)
✅ PASS  IsingXX(1.2) == RXX(1.2)
✅ PASS  IsingYY(0.5) == RYY(0.5)
✅ PASS  IsingYY(1.2) == RYY(1.2)
✅ PASS  IsingZZ(0.5) == RZZ(0.5)
✅ PASS  IsingZZ(1.2) == RZZ(1.2)


In [11]:
print("── Missing params raise ValueError ──────")
for canonical, _ in PARAM_2Q:
    try:
        Gate(canonical)._get_matrix()
        check(False, f"{canonical} without params should raise ValueError")
    except ValueError:
        check(True, f"{canonical} without params → ValueError")

── Missing params raise ValueError ──────
✅ PASS  RXX without params → ValueError
✅ PASS  RYY without params → ValueError
✅ PASS  RZZ without params → ValueError
✅ PASS  XXPlusYY without params → ValueError
✅ PASS  XXMinusYY without params → ValueError
✅ PASS  CRX without params → ValueError
✅ PASS  CRY without params → ValueError
✅ PASS  CRZ without params → ValueError


---
## 6  Channels

In [12]:
print("── AmplitudeDamping ─────────────────────")

# g=0 → identity
check(np.allclose(mat("AmplitudeDamping", [0.0]), np.eye(4), atol=ATol),
      "AmplitudeDamping(g=0) = I")

# g=1 → full decay: maps everything to |0⟩
AD1 = mat("AmplitudeDamping", [1.0])
check(preserves_trace(AD1), "AmplitudeDamping(g=1): trace-preserving")

# Non-unital: the shift vector c should be non-zero for g > 0
g = Gate("AmplitudeDamping", params=[0.5])
check(g._is_nonunital(), "AmplitudeDamping(0.5) is non-unital")

# Intermediate g: M[0,0] == 1 (trace preservation)
for gval in [0.1, 0.5, 0.9]:
    M = mat("AmplitudeDamping", [gval])
    check(np.isclose(M[0, 0], 1.0, atol=ATol), f"AmplitudeDamping(g={gval}): M[0,0]=1")

print("── Depolarizing ─────────────────────────")

# p=0 → identity
check(np.allclose(mat("Depolarizing", [0.0]), np.eye(4), atol=ATol),
      "Depolarizing(p=0) = I")

# p=1 → fully depolarised (off-diagonal Bloch components → 0)
Dep1 = mat("Depolarizing", [1.0])
check(np.allclose(Dep1[1:, 1:], np.zeros((3, 3)), atol=ATol),
      "Depolarizing(p=1): Bloch vector shrinks to 0")

# Should be unital
g = Gate("Depolarizing", params=[0.5])
check(not g._is_nonunital(), "Depolarizing is unital")

# Shrinkage factor must be in [0,1] for physical p∈[0,1]
for pval in [0.0, 0.25, 0.5, 0.75, 1.0]:
    M = mat("Depolarizing", [pval])
    shrink = M[1, 1]  # all off-diag equal
    check(0.0 <= float(shrink) <= 1.0 + ATol, f"Depolarizing(p={pval}): shrink∈[0,1]")

── AmplitudeDamping ─────────────────────
✅ PASS  AmplitudeDamping(g=0) = I
✅ PASS  AmplitudeDamping(g=1): trace-preserving
✅ PASS  AmplitudeDamping(0.5) is non-unital
✅ PASS  AmplitudeDamping(g=0.1): M[0,0]=1
✅ PASS  AmplitudeDamping(g=0.5): M[0,0]=1
✅ PASS  AmplitudeDamping(g=0.9): M[0,0]=1
── Depolarizing ─────────────────────────
✅ PASS  Depolarizing(p=0) = I
❌ FAIL  Depolarizing(p=1): Bloch vector shrinks to 0
✅ PASS  Depolarizing is unital
✅ PASS  Depolarizing(p=0.0): shrink∈[0,1]
✅ PASS  Depolarizing(p=0.25): shrink∈[0,1]
✅ PASS  Depolarizing(p=0.5): shrink∈[0,1]
✅ PASS  Depolarizing(p=0.75): shrink∈[0,1]
❌ FAIL  Depolarizing(p=1.0): shrink∈[0,1]


---
## 7  `expand_gate_Bloch` and `circuit_gate`

In [13]:
print("── Single-qubit expansion ───────────────")

H_1q = mat("H")   # 4×4

# Place H on qubit 0 of a 2-qubit system → H⊗I in Pauli basis
g = Gate("H", N_qubits=2, Gate_loc=0)
cg = np.array(g.circuit_gate())
check(cg.shape == (16, 16), "H on q0 of 2-qubit system: shape (16,16)")
check(is_orthogonal(cg), "H on q0 of 2-qubit system: orthogonal")

# Place H on qubit 1 of a 2-qubit system → I⊗H
g1 = Gate("H", N_qubits=2, Gate_loc=1)
cg1 = np.array(g1.circuit_gate())
check(is_orthogonal(cg1), "H on q1 of 2-qubit system: orthogonal")
check(not np.allclose(cg, cg1, atol=ATol), "H⊗I ≠ I⊗H")

# Applying H to both qubits independently then composed == H⊗H
HH = np.kron(H_1q, H_1q)
composed = cg1 @ cg  # H on q1 after H on q0
check(np.allclose(HH, composed, atol=ATol), "(I⊗H)·(H⊗I) = H⊗H")

print("── expand_gate_Bloch: non-adjacent qubits ──")
CNOT_mat = mat("CNOT")
# Control on q0, target on q2 in a 3-qubit system
expanded = np.array(expand_gate_Bloch(CNOT_mat, [0, 2], 3))
check(expanded.shape == (64, 64), "CNOT on [0,2] in 3-qubit system: shape (64,64)")
check(is_orthogonal(expanded), "CNOT on [0,2] in 3-qubit system: orthogonal")

print("── expand_gate_Bloch: reversed qubit order ──")
exp_fwd = np.array(expand_gate_Bloch(CNOT_mat, [0, 1], 2))
exp_rev = np.array(expand_gate_Bloch(CNOT_mat, [1, 0], 2))
check(not np.allclose(exp_fwd, exp_rev, atol=ATol), "CNOT[0,1] ≠ CNOT[1,0]")
check(is_orthogonal(exp_rev), "CNOT[1,0]: still orthogonal")

── Single-qubit expansion ───────────────
✅ PASS  H on q0 of 2-qubit system: shape (16,16)
✅ PASS  H on q0 of 2-qubit system: orthogonal
✅ PASS  H on q1 of 2-qubit system: orthogonal
✅ PASS  H⊗I ≠ I⊗H
✅ PASS  (I⊗H)·(H⊗I) = H⊗H
── expand_gate_Bloch: non-adjacent qubits ──
✅ PASS  CNOT on [0,2] in 3-qubit system: shape (64,64)
✅ PASS  CNOT on [0,2] in 3-qubit system: orthogonal
── expand_gate_Bloch: reversed qubit order ──
✅ PASS  CNOT[0,1] ≠ CNOT[1,0]
✅ PASS  CNOT[1,0]: still orthogonal


True

In [14]:
print("── circuit_gate location errors ─────────")

# Index out of range
try:
    Gate("H", N_qubits=2, Gate_loc=2).circuit_gate()
    check(False, "Gate_loc >= N_qubits should raise ValueError")
except (ValueError, IndexError):
    check(True, "Gate_loc >= N_qubits raises an error")

# Missing N_qubits
try:
    Gate("H", Gate_loc=0).circuit_gate()
    check(False, "Missing N_qubits should raise ValueError")
except ValueError:
    check(True, "Missing N_qubits raises ValueError")

# Missing Gate_loc
try:
    Gate("H", N_qubits=2).circuit_gate()
    check(False, "Missing Gate_loc should raise ValueError")
except ValueError:
    check(True, "Missing Gate_loc raises ValueError")

# Same index for 2-qubit gate
try:
    Gate("CNOT", N_qubits=2, Gate_loc=[0, 0]).circuit_gate()
    check(False, "CNOT with identical indices should raise ValueError")
except ValueError:
    check(True, "CNOT with identical indices raises ValueError")

── circuit_gate location errors ─────────
✅ PASS  Gate_loc >= N_qubits raises an error
✅ PASS  Missing N_qubits raises ValueError
✅ PASS  Missing Gate_loc raises ValueError
✅ PASS  CNOT with identical indices raises ValueError


---
## 8  `circuit_matrix`

In [15]:
print("── Bell-state preparation circuit ───────")
# |00⟩ → Bell state via H on q0, then CNOT[0,1]
bell_circuit = [
    Gate("H",    N_qubits=2, Gate_loc=0),
    Gate("CNOT", N_qubits=2, Gate_loc=[0, 1]),
]
C = np.real(np.array(circuit_matrix(bell_circuit, [])))
check(C.shape == (16, 16), "Bell circuit: shape (16,16)")
check(is_orthogonal(C),    "Bell circuit: orthogonal")

print("── Parameterised circuit ────────────────")
param_circuit = [
    Gate("RX", N_qubits=1, Gate_loc=0),
    Gate("RZ", N_qubits=1, Gate_loc=0),
]
C_param = np.real(np.array(circuit_matrix(param_circuit, [np.pi/4, np.pi/3])))
check(C_param.shape == (4, 4), "RX·RZ circuit: shape (4,4)")
check(is_orthogonal(C_param),  "RX·RZ circuit: orthogonal")

print("── Identity circuit (zero angles) ───────")
zero_circuit = [
    Gate("RX", N_qubits=1, Gate_loc=0),
    Gate("RY", N_qubits=1, Gate_loc=0),
    Gate("RZ", N_qubits=1, Gate_loc=0),
]
C_id = np.real(np.array(circuit_matrix(zero_circuit, [0.0, 0.0, 0.0])))
check(np.allclose(C_id, np.eye(4), atol=ATol), "All-zero RX·RY·RZ = I")

print("── 3-qubit circuit ──────────────────────")
three_q = [
    Gate("H",    N_qubits=3, Gate_loc=0),
    Gate("CNOT", N_qubits=3, Gate_loc=[0, 1]),
    Gate("CNOT", N_qubits=3, Gate_loc=[1, 2]),
]
C3 = np.real(np.array(circuit_matrix(three_q, [])))
check(C3.shape == (64, 64), "3-qubit GHZ circuit: shape (64,64)")
check(is_orthogonal(C3),    "3-qubit GHZ circuit: orthogonal")

── Bell-state preparation circuit ───────
✅ PASS  Bell circuit: shape (16,16)
✅ PASS  Bell circuit: orthogonal
── Parameterised circuit ────────────────
✅ PASS  RX·RZ circuit: shape (4,4)
✅ PASS  RX·RZ circuit: orthogonal
── Identity circuit (zero angles) ───────
✅ PASS  All-zero RX·RY·RZ = I
── 3-qubit circuit ──────────────────────
✅ PASS  3-qubit GHZ circuit: shape (64,64)
✅ PASS  3-qubit GHZ circuit: orthogonal


True

---
## 9  `calculate_steadystate_solve`

In [16]:
print("── Amplitude-damping steady state ───────")
# A single amplitude-damping channel iterated to steady state should give |0⟩
# In the coherence-vector picture the steady state has x=y=0, z=-1/√2 (for n=1)
ad_circuit = [Gate("AmplitudeDamping", N_qubits=1, Gate_loc=0, params=[0.9])]
ss = np.array(calculate_steadystate_solve(ad_circuit, []))
check(ss.shape == (3,), "AmplitudeDamping steady state: shape (3,)")
check(np.allclose(ss[:2], 0.0, atol=ATol), "Steady state x=y=0")
check(ss[2] < 0, "Steady state z < 0  (ground state)")

print("── Depolarising steady state ─────────────")
# Full depolarising (p→1) steady state is maximally mixed → Bloch vector = 0
dep_circuit = [Gate("Depolarizing", N_qubits=1, Gate_loc=0, params=[0.99])]
ss_dep = np.array(calculate_steadystate_solve(dep_circuit, []))
check(np.allclose(ss_dep, 0.0, atol=1e-3), "Depolarising steady state ≈ 0")

print("── Unitary circuit has no unique steady state (degenerate) ─")
# A purely unitary circuit (no noise) will make (I - Ω) singular.
# We just check that the function either returns something or raises a linear-algebra error.
import traceback
unitary_circuit = [Gate("H", N_qubits=1, Gate_loc=0)]
try:
    ss_u = calculate_steadystate_solve(unitary_circuit, [])
    check(True, "Unitary circuit: returned (may be numerically noisy)")
except Exception:
    check(True, "Unitary circuit: raised expected linear-algebra error")

print("── Parameterised channel: param consumed correctly ──")
ad_p_circuit = [Gate("AmplitudeDamping", N_qubits=1, Gate_loc=0)]
for gval in [0.1, 0.5, 0.9]:
    ss = np.array(calculate_steadystate_solve(ad_p_circuit, [gval]))
    check(ss.shape == (3,), f"AmplitudeDamping(g={gval}) steady state: shape ok")

── Amplitude-damping steady state ───────


IndexError: list index out of range

---
## 10  Density-matrix ↔ Pauli-vector round-trip

In [17]:
import functools as ft

def random_dm(n, seed=0):
    """Random valid n-qubit density matrix."""
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((2**n, 2**n)) + 1j * rng.standard_normal((2**n, 2**n))
    rho = A @ A.conj().T
    return rho / np.trace(rho)

print("── 1-qubit round-trip ───────────────────")
for seed in range(5):
    rho = random_dm(1, seed)
    v   = density_matrix_to_pauli_vector(jnp.array(rho))
    rho2 = pauli_vector_to_density_matrix(v, n_qubits=1)
    check(np.allclose(np.array(rho2), rho, atol=ATol), f"1-qubit round-trip (seed={seed})")

print("── 2-qubit round-trip ───────────────────")
for seed in range(3):
    rho = random_dm(2, seed)
    v   = density_matrix_to_pauli_vector(jnp.array(rho))
    rho2 = pauli_vector_to_density_matrix(v, n_qubits=2)
    check(np.allclose(np.array(rho2), rho, atol=ATol), f"2-qubit round-trip (seed={seed})")

print("── Known states ─────────────────────────")
# |0⟩⟨0|
rho0 = np.array([[1, 0], [0, 0]], dtype=complex)
v0 = np.array(density_matrix_to_pauli_vector(jnp.array(rho0)))
check(np.isclose(v0[0], 1/np.sqrt(2), atol=ATol), "|0⟩: v[0] = 1/√2")
check(np.isclose(v0[3], -1/np.sqrt(2), atol=ATol), "|0⟩: v[Z] = -1/√2")
check(np.allclose(v0[1:3], 0, atol=ATol), "|0⟩: v[X]=v[Y]=0")

# |+⟩⟨+|  (X eigenstate)
rhoP = np.array([[0.5, 0.5], [0.5, 0.5]], dtype=complex)
vP   = np.array(density_matrix_to_pauli_vector(jnp.array(rhoP)))
check(np.isclose(vP[1], 1/np.sqrt(2), atol=ATol), "|+⟩: v[X] = 1/√2")
check(np.allclose(vP[[2,3]], 0, atol=ATol), "|+⟩: v[Y]=v[Z]=0")

── 1-qubit round-trip ───────────────────
✅ PASS  1-qubit round-trip (seed=0)
✅ PASS  1-qubit round-trip (seed=1)
✅ PASS  1-qubit round-trip (seed=2)
✅ PASS  1-qubit round-trip (seed=3)
✅ PASS  1-qubit round-trip (seed=4)
── 2-qubit round-trip ───────────────────
✅ PASS  2-qubit round-trip (seed=0)
✅ PASS  2-qubit round-trip (seed=1)
✅ PASS  2-qubit round-trip (seed=2)
── Known states ─────────────────────────
✅ PASS  |0⟩: v[0] = 1/√2
❌ FAIL  |0⟩: v[Z] = -1/√2
✅ PASS  |0⟩: v[X]=v[Y]=0
✅ PASS  |+⟩: v[X] = 1/√2
✅ PASS  |+⟩: v[Y]=v[Z]=0


True

---
## 11  `pauli_basis_strings` and `string_to_vector`

In [18]:
print("── pauli_basis_strings ──────────────────")
for n in [1, 2, 3]:
    strings = pauli_basis_strings(n)
    check(len(strings) == 4**n,            f"n={n}: {4**n} basis strings")
    check(strings[0] == "I" * n,           f"n={n}: first string is {'I'*n}")
    check(len(set(strings)) == len(strings), f"n={n}: all strings unique")
    check(all(len(s) == n for s in strings), f"n={n}: all strings length {n}")

print("── string_to_vector ─────────────────────")
# 'I' should map to index 0
v_I = string_to_vector("I")
check(v_I[0] == 1.0 and np.sum(v_I) == 1.0, "'I' → index 0")

# 'Z' should map to index 3
v_Z = string_to_vector("Z")
check(v_Z[3] == 1.0 and np.sum(v_Z) == 1.0, "'Z' → index 3")

# Two-qubit strings
v_IZ = string_to_vector("IZ")
check(v_IZ[3] == 1.0 and np.sum(v_IZ) == 1.0, "'IZ' → index 3")

v_ZI = string_to_vector("ZI")
check(v_ZI[12] == 1.0 and np.sum(v_ZI) == 1.0, "'ZI' → index 12")

# Orthogonality: distinct basis vectors should be orthogonal
for s1, s2 in [("X", "Y"), ("XI", "IX"), ("ZZ", "XX")]:
    v1 = string_to_vector(s1)
    v2 = string_to_vector(s2)
    check(np.dot(v1, v2) == 0.0, f"'{s1}' ⊥ '{s2}'")

── pauli_basis_strings ──────────────────
✅ PASS  n=1: 4 basis strings
✅ PASS  n=1: first string is I
✅ PASS  n=1: all strings unique
✅ PASS  n=1: all strings length 1
✅ PASS  n=2: 16 basis strings
✅ PASS  n=2: first string is II
✅ PASS  n=2: all strings unique
✅ PASS  n=2: all strings length 2
✅ PASS  n=3: 64 basis strings
✅ PASS  n=3: first string is III
✅ PASS  n=3: all strings unique
✅ PASS  n=3: all strings length 3
── string_to_vector ─────────────────────
✅ PASS  'I' → index 0
✅ PASS  'Z' → index 3
✅ PASS  'IZ' → index 3
✅ PASS  'ZI' → index 12
✅ PASS  'X' ⊥ 'Y'
✅ PASS  'XI' ⊥ 'IX'
✅ PASS  'ZZ' ⊥ 'XX'


---
## 12  `n_params` and `_is_nonunital`

In [19]:
print("── n_params ─────────────────────────────")
zero_param = ["H", "X", "Y", "Z", "T", "S", "CNOT", "SWAP", "XI"]
one_param  = ["RX", "RY", "RZ", "RXX", "RYY", "RZZ",
              "CRX", "CRY", "CRZ", "AmplitudeDamping", "Depolarizing",
              "XXPlusYY", "XXMinusYY"]

for name in zero_param:
    check(Gate(name).n_params() == 0, f"{name}.n_params() == 0")

for name in one_param:
    check(Gate(name).n_params() == 1, f"{name}.n_params() == 1")

print("── _is_nonunital ────────────────────────")
unital_gates = [
    Gate("H"),
    Gate("X"),
    Gate("RX",  params=[0.7]),
    Gate("RZZ", params=[0.4]),
    Gate("CNOT"),
    Gate("Depolarizing", params=[0.3]),
]
for g in unital_gates:
    check(not g._is_nonunital(), f"{g.name}: unital")

nonunital_gates = [
    Gate("AmplitudeDamping", params=[0.5]),
    Gate("AmplitudeDamping", params=[0.9]),
]
for g in nonunital_gates:
    check(g._is_nonunital(), f"{g.name}(p={g.params[0]}): non-unital")

── n_params ─────────────────────────────
✅ PASS  H.n_params() == 0
✅ PASS  X.n_params() == 0
✅ PASS  Y.n_params() == 0
✅ PASS  Z.n_params() == 0
✅ PASS  T.n_params() == 0
✅ PASS  S.n_params() == 0
✅ PASS  CNOT.n_params() == 0
✅ PASS  SWAP.n_params() == 0
✅ PASS  XI.n_params() == 0
✅ PASS  RX.n_params() == 1
✅ PASS  RY.n_params() == 1
✅ PASS  RZ.n_params() == 1
✅ PASS  RXX.n_params() == 1
✅ PASS  RYY.n_params() == 1
✅ PASS  RZZ.n_params() == 1
✅ PASS  CRX.n_params() == 1
✅ PASS  CRY.n_params() == 1
✅ PASS  CRZ.n_params() == 1
✅ PASS  AmplitudeDamping.n_params() == 1
✅ PASS  Depolarizing.n_params() == 1
✅ PASS  XXPlusYY.n_params() == 1
✅ PASS  XXMinusYY.n_params() == 1
── _is_nonunital ────────────────────────
✅ PASS  H: unital
✅ PASS  X: unital
✅ PASS  RX: unital
✅ PASS  RZZ: unital
✅ PASS  CNOT: unital
✅ PASS  Depolarizing: unital
✅ PASS  AmplitudeDamping(p=0.5): non-unital
✅ PASS  AmplitudeDamping(p=0.9): non-unital


---
## 13  Summary

In [20]:
print("All cells executed without exceptions — check ✅/❌ lines above for individual results.")

All cells executed without exceptions — check ✅/❌ lines above for individual results.
